In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,ArrayType, FloatType

from pyspark.sql import functions as F

###Question 1. Write a Spark code snippet to calculate the sum of a column in a DataFrame

In [0]:
data = [("John Doe", "john@example.com", 50000.0),
    ("Jane Smith", "jane@example.com", 60000.0),
    ("Bob Johnson", "bob@example.com", 55000.0)]

schema =StructType([
    StructField("name",StringType()),
    StructField("email",StringType()),
    StructField("salary",DoubleType())
])

df = spark.createDataFrame(data,schema)
df.display()

In [0]:
df_final = df.agg(F.sum(F.col("salary"))).first()[0]
df_final

###Question 2. Get all the Marks for individual in list

In [0]:
data=[(1,'Watson',34),(1,'Watson',40),(1,'Watson',34),(2,'Alex',45),(2,'Alex',50)]
schema=StructType([
    StructField("ID",IntegerType()),
    StructField("Name",StringType()),
    StructField("Marks",IntegerType())
])
df=spark.createDataFrame(data,schema)
df.display()

In [0]:
df_final = df.groupBy("ID","name").agg(F.collect_list("marks").alias("marks_list"))
df_final.display()

###Question 3: Identify rows containing non-numeric values in the “Quantity” column, if any.

In [0]:
schema = StructType([
  StructField("ProductCode", StringType(), True),
  StructField("Quantity", StringType(), True),
  StructField("UnitPrice", StringType(), True),
  StructField("CustomerID", StringType(), True),
])
 
data = [
  ("Q001", 5, 20.0, "C001"),
  ("Q002", 3, 15.5, "C002"),
  ("Q003", 10, 5.99, "C003"),
  ("Q004", 2, 50.0, "C001"),
  ("Q005", "nein", 12.75, "C002"),
]

df = spark.createDataFrame(data,schema)
df = df.display()

In [0]:
# Recreate df from Cell 9 since display() returned None
df = spark.createDataFrame(data, schema)
df_3 = df.withColumn("has_non_numeric", F.when(F.col("Quantity").rlike(r"[^0-9]"), True).otherwise(False))
df_3.display()

###Questions 4. Write PySpark Code to get the below output.
Input:<br>
| Name  | Hobbies |<br>

| Paris |   Polo,Tennis |<br>
| Matt  |   Golf, Hockey|<br>
| Sam   |   null |<br>

Output:<br>
| Name  | col|<br>

| Paris | Polo|<br>
| Paris | Tennis |<br>
| Matt  | Golf|<br>
| Matt  | Hockey|<br>


In [0]:
data = [
    ("Paris",["Polo","Tennis"]),
    ("Matt",["Golf","Hockey"]),
    ("Sam",[]),
]

schema = StructType([
    StructField("name",StringType()),
    StructField("sports",ArrayType(StringType()))
])

df = spark.createDataFrame(data,schema)
df.display()

# the arraytype column also be given as comma seperated string 
data2 = [
    ("Paris","Polo, Tennis"),
    ("Matt","Golf, Hockey"),
    ("Sam",""),
]

schema2 = StructType([
    StructField("name",StringType()),
    StructField("sports",StringType())
])

df2= spark.createDataFrame(data2,schema2)
df2.display()

In [0]:
df_final = df.withColumn("col", F.explode(F.col("sports")))
df_final.select("name","col").show()

# the array is given as string the just split them first and use explode on the new splitted list
df_final2 = df2.withColumn("game", F.split(F.col("sports"),","))
df_final2 = df_final2.withColumn("col", F.explode(F.col("game")))
df_final2.select("name","col").show()

###Question 5. Find the year, start_week_date, end_week_date, week_num

In [0]:
data=[(2025,1,'2025-01-01'),
      (2025,1,'2025-01-02'),
      (2025,1,'2025-01-03'),
      (2025,1,'2025-01-04'),
      (2025,1,'2025-01-05'),
      (2025,1,'2025-01-06'),
      (2025,1,'2025-01-07'),
      (2025,2,'2025-01-08'),
      (2025,2,'2025-01-09'),
      (2025,2,'2025-01-10'),
      (2025,2,'2025-01-11'),
      (2025,2,'2025-01-12'),
      (2025,2,'2025-01-13'),
      (2025,2,'2025-01-14')]

schema=StructType([StructField('year',IntegerType(),True),
                   StructField('week_num',IntegerType(),True),
                   StructField('dates',StringType(),True)])
df=spark.createDataFrame(data,schema)
df.display()

In [0]:
df = df.withColumn("dates",F.to_date(F.col("dates"),"yyyy-MM-dd"))
df_final = df.groupBy("year","week_num").agg(F.min(F.col("dates")).alias("start_week_date"),F.max(F.col("dates")).alias("end_week_date")
                                             )
df_final.display()

###Questions 6. Write a program to return the shipped and delivered rate for each order. Return order_id, shipped percentage, and delivered percentage.

In [0]:
data = [
 (1, "Shipped", "2025-04-01"),
 (1, "Shipped", "2025-04-02"),
 (2, "Delivered", "2025-04-01"),
 (2, "Delivered", "2025-04-02"),
 (2, "Shipped", "2025-04-03"),
 (3, "Shipped", "2025-04-01"),
 (3, "Delivered", "2025-04-02"),
 (3, "Delivered", "2025-04-03")
]

schema = StructType([
 StructField("order_id", IntegerType(), False),
 StructField("order_status", StringType(), True),
 StructField("order_date", StringType(), True)
])

orders_df = spark.createDataFrame(data, schema)
display(orders_df)

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("order_id")
df_6 = orders_df.withColumn("total_orders",F.count("order_id").over(window_spec))\
    .withColumn("total_status",F.count("order_status").over(window_spec))
df_6 = df_6.select("order_id","total_orders","total_status")

display(df_6)

In [0]:
df_6 = orders_df.withColumn("is_shipped", F.when(F.col("order_status")=='Shipped',1).otherwise(0))\
    .withColumn("is_delivered", F.when(F.col("order_status")=='Delivered',1).otherwise(0))
df_6 = df_6.groupBy("order_id").agg(F.count("*").alias("total_orders"),
                                    F.sum("is_shipped").alias("total_shipped"),
                                    F.sum("is_delivered").alias("total_delivered"))
df_6 = df_6.withColumn("shipped_percentage",F.round(F.col("total_shipped")/F.col("total_orders")*100,2))\
    .withColumn("Delivered_percentage",F.round(F.col("total_delivered")/F.col("total_orders")*100 , 2))\
    .select("order_id","shipped_percentage","Delivered_percentage")
df_6.display()

###7. Find the top most department having highest salary

In [0]:
from pyspark.sql import Row
schema = StructType([
StructField("id", IntegerType(), nullable=False),
StructField("name", StringType(), nullable=False), 
StructField("age", IntegerType(), nullable=False),
StructField("department", StringType(), nullable=False), 
StructField("salary", DoubleType(), nullable=False)
])
data = [
    Row(1, "John", 30, "Sales", 50000.0),
    Row(2, "Alice", 28, "Marketing", 60000.0),
    Row(3, "Bob", 32, "Finance", 55000.0),
    Row(4, "Sarah", 29, "Sales", 52000.0),
    Row(5, "Mike", 31, "Finance", 58000.0)
]
employeeDF = spark.createDataFrame(data, schema)
display(employeeDF)

In [0]:
from pyspark.sql.window import Window
df_7 = employeeDF.groupBy("department").agg(F.sum(F.col("salary"))).alias("total_salary")
df_7.display()
# df_7 = df_7.withColumn("max_salary",)

df_7 = employeeDF.withColumn("total_salary", F.sum("salary").over(Window.partitionBy("department")))
df_7 =  df_7.select("department","total_salary").orderBy(F.col("total_salary").desc()).limit(1)

df_7.display()

# employeeDF_1=employeeDF.groupBy("department").agg(sum(col("salary")).alias("total_salary"))
# window_spec=Window.orderBy(col("total_salary").desc())
# df_final=employeeDF_1.withColumn("rn",row_number().over(window_spec)).filter(col("rn")==1).select("department")
# display(df_final)

# df_final=employeeDF.groupBy("department").agg(sum("salary").alias("total_salary")).orderBy(col("total_salary").desc()).limit(1)
# display(df_final)

###Questions: 8. Find the origin and the destination(last flight) flights for each customer.

In [0]:
flights_data = [(1,'Flight2' , 'Los Angeles' , 'London'),
(1,'Flight1' , 'London' , 'Vatican'),
(1,'Flight3' , 'Vatican' , 'Nantes'),
(2,'Flight1' , 'Houston' , 'Paris'),
(2,'Flight2' , 'Paris' , 'Nice')
]

schema = "cust_id int, flight_id string , origin string , destination string"
flights_df = spark.createDataFrame(flights_data,schema)
display(flights_df)

In [0]:
from pyspark.sql.window import Window
window_spec= Window.partitionBy("cust_id").orderBy("flight_id")
df_8 = flights_df.withColumn("first_flight", F.first('origin').over(window_spec))\
        .withColumn("last_flight", F.last('destination').over(window_spec))

df_8 = df_8.select("cust_id","first_flight","last_flight").groupby("cust_id").agg(F.first("first_flight"),F.last("last_flight"))

# df_8 = flights_df.orderBy("cust_id","flight_id").groupBy("cust_id").agg(F.first("origin"),F.last("destination"))   
df_8.display()

###Questions 9. Write a PySpark program to select every 3rd (nth) row in the dataset

In [0]:
from pyspark.sql.window import Window

schema = StructType([
 StructField("emp_id", IntegerType(), True),
 StructField("name", StringType(), True),
 StructField("salary", IntegerType(), True)
])

data = [
 (1001, "John Doe", 50000),
 (2001, "Jane Smith", 60000),
 (1003, "Michael Johnson", 75000),
 (4000, "Emily Davis", 55000),
 (1005, "Robert Brown", 70000),
 (6000, "Emma Wilson", 80000),
 (1700, "James Taylor", 65000),
 (8000, "Olivia Martinez", 72000),
 (2900, "William Anderson", 68000),
 (3310, "Sophia Garcia", 67000)
]

df = spark.createDataFrame(data, schema)

df.display()

In [0]:
window_spec = Window.orderBy(F.lit('1'))
df_9 =df.withColumn("rn", F.row_number().over(window_spec))
df_9 = df_9.filter(F.col("rn")%3 ==0 )
df_9.display()

###Question 10. Handle Data that has Error

In [0]:
data = [
 ('2025-01-01', '$100.50'),
 ('Jan 15, 2025', '€169.75'),
 ('03/20/25', '£150.20'),
 ('2025-05-13', '¥300.00')
]
columns = ['date', 'amount']

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Print the original DataFrame
print("Original DataFrame:")
display(df)


In [0]:
df_10 =df.withColumn("date1",F.coalesce(
    F.try_to_date(F.col("date"),'yyyy-MM-dd'),
    F.try_to_date(F.col("date"),'MM/dd/yy'),
    F.try_to_date(F.col("date"),'MMM dd, yyyy')
    ))\
    .withColumn("amount",F.regexp_replace(F.col("amount"), '[^0-9.]','').cast('double'))
df_10.display()

###Question 11. Get the below output by using pyspark
ID col1 col2<br>
1 A 10<br>
1 B 20<br>
2 A 30<br>
2 B 40<br><br>
Output:<br>
ID A B<br>
1 10 20<br>
2 30 40<br>

In [0]:
data=[(1,'A',10),(2,'A',20),(1,'B',30),(2,'B',40)]
schema="ID int,col1 string,col2 int"
df = spark.createDataFrame(data,schema)
display(df)

In [0]:
columns =['A','B']
df_11 = df.groupBy("ID").pivot("col1", values=columns).agg(F.sum(F.col("col2")))
df_11.display()
df_11b = df.groupBy("ID").pivot("col1", values=columns).agg(F.first(F.col("col2")))
df_11b.display()

###Questions 12. Calculate the sum of salaries for employees whose names start with the letter “J”

In [0]:
from pyspark.sql import Row
schema = StructType([
StructField("id", IntegerType(), nullable=False),
StructField("name", StringType(), nullable=False), 
StructField("age", IntegerType(), nullable=False),
StructField("department", StringType(), nullable=False), 
StructField("salary", DoubleType(), nullable=False)
])
data = [
    Row(1, "John", 30, "Sales", 50000.0),
    Row(2, "Alice", 28, "Marketing", 60000.0),
    Row(3, "Bob", 32, "Finance", 55000.0),
    Row(4, "Kevin", 29, "Sales", 52000.0),
    Row(5, "Mike", 31, "Finance", 58000.0)
]
employeeDF = spark.createDataFrame(data, schema)
display(employeeDF)

In [0]:
df_12 = employeeDF.filter(F.upper(F.col("name")).startswith('K%'))

    # .agg(F.sum(F.col("salary")))
df_12.display()

###Question 13. Find the first non null phone number for the customer

In [0]:
data=[(1, "90909090900", None, None),
 (2, None, "8893029324903", "79767679988"),
 (3, None, None, "9090897576"),
 (4, None, None, None)
]

columns = ["customer_id", "home_phone", "mobile_phone", "work_phone"]

df = spark.createDataFrame(data, columns)
display(df)


In [0]:
df13 = df.withColumn("first_non_null",F.coalesce(F.col("home_phone"),F.col("mobile_phone"),F.col("work_phone"))).drop(F.col("home_phone"),F.col("mobile_phone"),F.col("work_phone"))
df13.display()

###Question 14. find id which is even number and description is meh

In [0]:

# Define schema for the DataFrame
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("movie", StringType(), True),
    StructField("description", StringType(), True),
    StructField("rating", FloatType(), True)
])

# Create data for the DataFrame
data = [
    (1, "War", "great 3D", 8.9),
    (2, "Science", "fiction", 8.5),
    (3, "French", "meh", 6.2),
    (4, "Ice song", "Fantacy", 8.6),
    (5, "House card", "Interesting", 9.1),
    (6, "French", "meh", 9.2)
]

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Show the DataFrame
df.display()

In [0]:
df14 = df.filter("id % 2 == 0 and description = 'meh' ")
df14.display()